# Laboratorio interactivo 1: Exploración de datos financieros con Big Data

## 1. Introducción
El objetivo de este laboratorio es familiarizarse con el flujo básico de análisis de datos financieros. Aquí aprenderás a cargar datos, explorarlos, realizar tareas de limpieza y transformación, visualizar las series de precios y retornos, calcular indicadores financieros sencillos y finalmente construir un primer modelo predictivo de regresión lineal para intentar pronosticar el retorno del siguiente periodo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuracion de graficas
plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Carga de un dataset financiero
Al no disponer de un archivo local, generaremos datos simulados que representen el precio de cierre de un activo a lo largo del tiempo.

In [ ]:
# Generacion de datos simulados
np.random.seed(42)
fechas = pd.date_range(start="2022-01-01", periods=500, freq="B")
retornos_sim = np.random.normal(loc=0.0001, scale=0.015, size=500)
precios = 100 * np.exp(np.cumsum(retornos_sim))

df = pd.DataFrame({
    "Fecha": fechas,
    "Precio_Cierre": precios
})
# Introducimos algunos NaN intencionalmente para la limpieza
df.loc[10:12, "Precio_Cierre"] = np.nan
df.head()

## 3. Lectura y exploración inicial de los datos
Vamos a verificar la información del dataframe, el tipo de datos y si existen valores nulos.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Deteccion de valores faltantes
print("Valores nulos por columna:\n", df.isnull().sum())

## 4. Limpieza básica
Lidiaremos con los NaN utilizando interpolación o llenado hacia adelante. Además, prepararemos el índice de fechas y calcularemos los retornos.

In [ ]:
# Convertir a fecha y fijar como indice
df["Fecha"] = pd.to_datetime(df["Fecha"])
df.set_index("Fecha", inplace=True)

# Manejo de NaN: Llenamos los valores nulos con el metodo forward fill (ultimo valor conocido)
df["Precio_Cierre"] = df["Precio_Cierre"].ffill()

# Retorno simple
df["Retorno_Simple"] = df["Precio_Cierre"].pct_change()

# Retorno logaritmico (R_i = ln(P_t / P_{t-1}))
df["Retorno_Log"] = np.log(df["Precio_Cierre"] / df["Precio_Cierre"].shift(1))

df.dropna(inplace=True) # Eliminar la primera fila que ahora tiene NaN en retornos
df.head()

## 5. Visualización inicial
Visualizamos la serie de precios y los retornos.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 12), gridspec_kw={"height_ratios": [2, 1, 1]})

# Grafico de precios
ax[0].plot(df.index, df["Precio_Cierre"], color="blue")
ax[0].set_title("Precio de Cierre Historico")
ax[0].set_ylabel("Precio")

# Grafico de retornos
ax[1].plot(df.index, df["Retorno_Log"], color="gray", linewidth=0.8)
ax[1].set_title("Retornos Logaritmicos")
ax[1].set_ylabel("Retorno")

# Histograma de retornos
ax[2].hist(df["Retorno_Log"], bins=50, color="orange", edgecolor="black")
ax[2].set_title("Distribucion de Retornos Logaritmicos")

plt.tight_layout()
plt.show()

## 6. Cálculo de indicadores simples
Agregaremos volatilidad móvil, medias móviles y el retorno acumulado.

In [ ]:
# Volatilidad (Desviacion estandar de 20 dias)
df["Volatilidad_20d"] = df["Retorno_Log"].rolling(window=20).std()

# Medias Moviles
df["SMA_20"] = df["Precio_Cierre"].rolling(window=20).mean()
df["SMA_50"] = df["Precio_Cierre"].rolling(window=50).mean()

# Retorno Acumulado
df["Retorno_Acumulado"] = (1 + df["Retorno_Simple"]).cumprod() - 1

df.tail()

## 7. Modelo predictivo sencillo
Utilizaremos regresión lineal para predecir el retorno del día siguiente usando información de los retornos pasados (lags) y la volatilidad.

In [ ]:
# Creacion de variables predictoras (Lags)
df["Lag_1"] = df["Retorno_Log"].shift(1)
df["Lag_2"] = df["Retorno_Log"].shift(2)

# Variable objetivo: El retorno del periodo siguiente
df["Target"] = df["Retorno_Log"].shift(-1)

# Eliminar NaNs resultantes
df_model = df.dropna()

# Definicion de X e y
features = ["Lag_1", "Lag_2", "Volatilidad_20d"]
X = df_model[features]
y = df_model["Target"]

# Division en train y test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Entrenamiento del modelo
model = LinearRegression()
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

# Evaluacion de metricas
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.5f}")
print(f"MSE: {mse:.5f}")
print(f"R2: {r2:.5f}")

## 8. Conclusiones del laboratorio
- Hemos limpiado y transformado exitosamente datos crudos en métricas financieras estándar.
- La visualización nos permite observar el comportamiento de la serie y la naturaleza leptocúrtica de los retornos.
- El modelo de regresión lineal muestra un $R^2$ cercano a cero o negativo, lo que ilustra lo expuesto en el capítulo teórico: los modelos simples basados solo en la acción del precio pasada (sin validación estricta ni selección de variables causales) tienen una capacidad predictiva limitada debido al ruido inherente de los mercados financieros.